# Predict, then Select — pretrained models

Evaluate the published CNN–ViT checkpoints on complete [BSCCMNIST](https://doi.org/10.5061/dryad.sxksn038s) candidate pools. The default uses all **1,358 within-slide test cells**, CD16, and all three training seeds. No training or model selection is performed.

Choose a GPU runtime for faster inference, then run all cells. Only the prepared cohort and model package needed for your settings are downloaded. The default is approximately 140 MB, including optional reference results. [Methods and reproduction](https://github.com/yyttim/predict-then-select).

In [ ]:
# Install the published inference code.
import subprocess, sys
COMMIT = "ad6422d16c01e1cb5a478466092f54161422cde1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown==6.4.2",
    f"cellselect @ git+https://github.com/yyttim/predict-then-select.git@{COMMIT}"])

import hashlib, json, shutil, stat, zipfile
from pathlib import Path, PurePosixPath
import gdown
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from cellselect.data import load_prepared
from cellselect.external_data import load_external
from cellselect.model import build_model
from cellselect.training import predict
from cellselect.study import class_mean_scores, study_prediction_metrics, outcome_metrics

In [ ]:
# Experiment and downloads.
marker = "CD16"  # @param ["CD16", "CD45"]
train_fraction = 1.0  # @param [0.25, 0.5, 1.0] {type:"raw"}
pool = "within_slide_test"  # @param ["within_slide_test", "external0", "external1"]
q = 0.10  # @param [0.05, 0.10, 0.20] {type:"raw"}
compare_reference = True  # @param {type:"boolean"}
show_plot = True  # @param {type:"boolean"}
ROOT = Path("predict_then_select_release").resolve()
ARTIFACTS = {
    "main_data": {
        "filename": "main-data.zip",
        "id": "1dH0RB-eDWQFSCgOa3S8R4WzG_1tTpXL4",
        "sha256": "6a22ec38b5039874965b42678c4d6a37792257e5fd2c2f462cdb96e260c60a67"
    },
    "external_data": {
        "filename": "external-data.zip",
        "id": "19LHIygsEloioxzvESuipC2hqUWRASam8",
        "sha256": "6aeb53fb381d4eefaac2806c5c0bf60a7f73d62445a3da3e58da68e22b35983a"
    },
    "cd16_full": {
        "filename": "cd16-full.zip",
        "id": "1syMMVg-7C_ywbbA5DOA-Jl8k-nx7YpEz",
        "sha256": "5558c61f2196aaf6d4b9b8258b21b8c47daa907f84819b1e168bd8519cd72b41"
    },
    "cd16_half": {
        "filename": "cd16-half.zip",
        "id": "17_OZfPeQauzT-7cGrr5htcx-3agxFC6z",
        "sha256": "f347e66334779eeabe114003cc2106aa224a0bb3609c6c03333dff8aefa57a50"
    },
    "cd16_quarter": {
        "filename": "cd16-quarter.zip",
        "id": "1ZDi3oMxxXzCk_pw85RtaT8DgMEk8G5YP",
        "sha256": "5e1196a28088f358619b5dc7c77a84974b272a2cda4af90cf3460a7ffb8d9f6c"
    },
    "cd45_full": {
        "filename": "cd45-full.zip",
        "id": "1sASoRNNAI1-aZzYPSzmSVb1ex4US2BnG",
        "sha256": "7cf978e50405abdc62a70c7581979439309e937b231e216455abb061cb441189"
    },
    "results": {
        "filename": "evaluation-results.zip",
        "id": "1GftHW2W_sAFEWbVBoSQm_h7ZETJl4Ve6",
        "sha256": "13fe1cc3b2469b379fc6ee90e30efa17a80e5ee87c41e3a692f8267ec5f6b92c"
    }
}
train_fraction, q = float(train_fraction), float(q)
assert marker in ("CD16", "CD45") and train_fraction in (0.25, 0.5, 1.0)
assert q in (0.05, 0.10, 0.20) and pool in ("within_slide_test", "external0", "external1")
assert marker != "CD45" or train_fraction == 1.0, "CD45 uses the full training cohort."
assert pool == "within_slide_test" or train_fraction == 1.0, "External evaluation uses full-data models."
model_package = "cd45_full" if marker == "CD45" else {0.25: "cd16_quarter", 0.5: "cd16_half", 1.0: "cd16_full"}[train_fraction]
packages = ["main_data", model_package]
if pool != "within_slide_test":
    packages.append("external_data")
if compare_reference:
    packages.append("results")

In [ ]:
# Verify each archive before extracting paths inside the release directory.
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def checked_path(name):
    relative = PurePosixPath(name)
    if relative.is_absolute() or ".." in relative.parts or "\\" in name or ":" in name:
        raise ValueError(f"Unsafe archive path: {name}")
    path = ROOT.joinpath(*relative.parts).resolve()
    if not path.is_relative_to(ROOT):
        raise ValueError(f"Path leaves the release directory: {name}")
    return path

ROOT.mkdir(parents=True, exist_ok=True)
for package in packages:
    artifact = ARTIFACTS[package]
    prefix = "data" if package.endswith("_data") else "results" if package == "results" else "models"
    expected = artifact["sha256"]
    if len(expected) != 64 or any(c not in "0123456789abcdef" for c in expected):
        raise ValueError("Invalid archive SHA-256.")
    archive = ROOT / artifact["filename"]
    if not archive.exists():
        gdown.download(id=artifact["id"], output=str(archive), use_cookies=False, quiet=False)
    if sha256(archive) != expected:
        raise ValueError(f"Checksum mismatch: {archive.name}")
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            if (PurePosixPath(member.filename).parts[0] != prefix
                    or stat.S_ISLNK(member.external_attr >> 16)):
                raise ValueError(f"Unexpected archive entry: {member.filename}")
            target = checked_path(member.filename)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(member) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)
print(f"{len(packages)} required archives verified and extracted.")

In [ ]:
# Load a complete pool and the validation-selected states.
index = json.loads((ROOT / "models/index.json").read_text())
assert index["code_commit"] == COMMIT
main_root = ROOT / "data/prepared/bsccmnist_cd16_v1"
assert sha256(main_root / "manifest.json") == index["data_manifest_sha256"]
main = load_prepared(main_root, verify_hashes=True)
if pool == "within_slide_test":
    dataset, positions, pool_id = main, main.rows("test"), pool
else:
    external_root = ROOT / "data/prepared/bsccmnist_batch1_all_v1"
    assert sha256(external_root / "manifest.json") == "078e19ce5ef828e3ba32ed29edce1575226645b5f3b27b2caa6929e2ab49397a"
    dataset = load_external(external_root, verify_hashes=True)
    pool_id = f"antibodies_all/batch_1/slide_replicate_{int(pool[-1])}"
    positions = dataset.pool_rows(pool_id)
meta = dataset.metadata.iloc[positions].reset_index(drop=True)
images = torch.from_numpy(np.asarray(dataset.images[positions], dtype=np.float32).copy())
ids, outcomes = meta.global_index.to_numpy(), meta[marker].to_numpy(dtype=np.float64)
records = [r for r in index["primary"]
           if r["marker"] == marker and r["train_fraction"] == train_fraction]
assert records and len({r["path"] for r in records}) == len(records)
policies = sorted({p for r in records for p in r["policies"]})
for policy in policies:
    assert sorted(r["seed"] for r in records if policy in r["policies"]) == [16, 26, 36]
for record in records:
    assert sha256(checked_path(record["path"])) == record["sha256"]
print(f"{marker}, training fraction {train_fraction:g}, {pool_id}: {len(meta):,} cells")
print(f"{len(records)} distinct checkpoints, {len(policies)} policies, three seeds each")

In [ ]:
# Each checkpoint is evaluated once; policy aliases reuse its predictions.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
rows = []
for record in records:
    checkpoint = torch.load(checked_path(record["path"]), map_location="cpu", weights_only=True)
    config, norm = checkpoint["config"], checkpoint["normalization"]
    assert checkpoint["manifest_sha256"] == index["data_manifest_sha256"]
    assert checkpoint["epoch"] == record["epoch"]
    assert (config["marker"], config["train_fraction"], config["seed"], config["mode"]) == (
        marker, train_fraction, record["seed"], record["mode"])
    model = build_model(feature_dim=config["feature_dim"]).to(device)
    model.load_state_dict(checkpoint["model"], strict=True)
    mean = torch.tensor(norm["image_mean"], device=device, dtype=torch.float32)[None, :, None, None]
    std = torch.tensor(norm["image_std"], device=device, dtype=torch.float32)[None, :, None, None]
    logits, prediction = predict(model, images, mean, std, device, batch_size=512, amp=False)
    score = (class_mean_scores(logits, norm["class_means"]) if record["mode"] == "classifier"
             else prediction.astype(np.float64) * norm["target_std"] + norm["target_mean"])
    calibration = record["calibration"]
    assert calibration["fit_partition"] == "val" and calibration["slope"] > 0
    calibrated = calibration["slope"] * score + calibration["intercept"]
    values = {"rmse": study_prediction_metrics(outcomes, score)["rmse"],
              "calibrated_rmse": study_prediction_metrics(outcomes, calibrated)["rmse"],
              "utility": outcome_metrics(outcomes, score, ids, q)["utility"]}
    for policy in record["policies"]:
        rows.append({"policy": policy, "seed": record["seed"], **values})
    del model
metrics = pd.DataFrame(rows)
print(f"Finished inference on {device}.")

In [ ]:
# Mean ± sample SD across the three trained seeds, in marker abundance units.
columns = ["rmse", "calibrated_rmse", "utility"]
means = metrics.groupby("policy")[columns].mean()
sds = metrics.groupby("policy")[columns].std(ddof=1)
labels = {"regression": "MSE", "joint": "MSE + CE", "spo_plus": "SPO+", "pg_central": "PG",
          "class_means": "Class means", "predictive_selector": "RMSE selection",
          "utility_selector": "Utility selection"}
order = list(labels)
means, sds = means.reindex(order), sds.reindex(order)
table = means.copy().astype(object)
for column in columns:
    table[column] = [f"{m:.3f} ± {s:.3f}" for m, s in zip(means[column], sds[column])]
display(table.rename(index=labels, columns={"rmse": "RMSE", "calibrated_rmse": "Calibrated RMSE",
                              "utility": f"Top-{q:.0%} mean {marker}"}))
if compare_reference:
    reference = pd.DataFrame(json.loads((ROOT / "results/reference/extended/results.json").read_text())["metrics"])
    reference = reference.loc[(reference.marker == marker) & (reference.train_fraction == train_fraction)
                              & (reference.pool == pool_id) & np.isclose(reference.fraction, q)]
    paired = metrics.merge(reference[["policy", "seed", *columns]], on=["policy", "seed"],
                           suffixes=("", "_reference"), validate="one_to_one")
    assert len(paired) == len(metrics), "Missing reference result."
    display(pd.Series({c: (paired[c] - paired[c + "_reference"]).abs().max() for c in columns},
                      name="Maximum absolute difference from reference"))
if show_plot:
    fig, ax = plt.subplots(figsize=(7, 3.7))
    ax.barh([labels[p] for p in order], means["utility"], xerr=sds["utility"],
            color="#48728c", capsize=3)
    ax.invert_yaxis()
    ax.axvline(outcomes.mean(), color="0.5", linestyle="--", label="Random selection")
    ax.set(xlabel=f"Top-{q:.0%} mean {marker} (arbitrary units)", ylabel="")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower right", bbox_to_anchor=(1, 1.01))
    plt.tight_layout()
    plt.show()


Scores use images and the original training normalization. The classifier baseline combines predicted class probabilities with training-set marker means. Measured markers enter evaluation only; true cell types are not supplied to the selection rule. The positive affine calibration was fitted on validation data and affects reported RMSE, not selection.

The within-slide pool was reused during project development, so its follow-up comparisons are exploratory. External slides were evaluated without fitting or reselection; compare methods within each slide because their measurement scales and populations differ. The displayed SD describes three trained seeds, not a confidence interval. Published field-bootstrap intervals condition on the frozen models and validation choices. Small numerical differences across hardware and library versions can change nearly tied selections.